In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

c:\Users\Annisa Tazkiya\Documents\chinese-indonesian-product-classification\.venv\Lib\site-packages\huggingface_hub\constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [3]:
df = pd.read_csv('../data/raw/products.csv')

In [4]:
x = df['product']
y = df['category']

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [5]:
model_name = 'xlm-roberta-base'

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
labels = sorted(df['category'].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

y_train_encoded = y_train.map(label2id)
y_test_encoded = y_test.map(label2id)

In [8]:
train_encodings = tokenizer(
    x_train.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    x_test.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

In [9]:
for text in x_test.head(5):
    print(text)
    print(tokenizer.tokenize(text))
    print()

便携式投影仪
['▁', '便', '携', '式', '投影', '仪']

map tulang ukuran A4
['▁map', '▁tulang', '▁ukuran', '▁A', '4']

lampu sorot tanam untuk koridor
['▁lampu', '▁so', 'rot', '▁tan', 'am', '▁untuk', '▁koridor']

buku agenda ukran A5
['▁buku', '▁agenda', '▁ukr', 'an', '▁A', '5']

baut kecil segi enam
['▁baut', '▁kecil', '▁segi', '▁enam']



In [10]:
class ProductDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels.ilox[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [11]:
train_dataset = ProductDataset(
    train_encodings,
    y_train_encoded.reset_index(drop=True)
)

test_dateset = ProductDataset(
    test_encodings,
    y_test_encoded.reset_index(drop=True)
)

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(labels), 
    label2id=label2id, 
    id2label=id2label,
    low_cpu_mem_usage=True
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
from transformers import TrainingArguments, Trainer

In [16]:
training_args = TrainingArguments(
    output_dir='./xlmr_results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to='none'
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [17]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
pip install --upgrade pip

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2
Note: you may need to restart the kernel to use updated packages.
